In [ ]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import os

# scifact
# arguana
# nfcorpus
# scidocs

def data_loader(dataset: str):
    url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"
    data_path = util.download_and_unzip(url, "data")

    corpus_path = os.path.join(data_path, "corpus.jsonl")
    queries_path = os.path.join(data_path, "queries.jsonl")
    qrels_path = os.path.join(data_path, "qrels", "test.tsv")

    corpus, queries, qrels = GenericDataLoader(
        corpus_file=corpus_path,
        query_file=queries_path,
        qrels_file=qrels_path
    ).load_custom()
    return corpus, queries, qrels


In [ ]:
import spacy

def tokenize(s):
    nlp = spacy.load("en_core_web_sm")
    s = nlp(s)
    tokens = [token.text.lower()
              for token in s
              if not token.is_punct and not token.is_stop and not token.is_space]
    return tokens

def clean_data(corpus: dict, queries: dict):
    cleaned_corpus = {}
    for doc_id, doc in corpus.items():
        title = doc.get("title", "")
        text = doc.get("text", "")
        full_text = title + " " + text
        tokens = tokenize(full_text)
        cleaned_corpus[doc_id] = {"text": " ".join(tokens)}

    cleaned_queries = {}
    for query_id, query_text in queries.items():
        tokens = tokenize(query_text)
        cleaned_queries[query_id] = {"text": " ".join(tokens)}

    return cleaned_corpus, cleaned_queries

c:\Users\Axel\Desktop\lsa rocchio knn\BEIR_lsa_rocchio_knn\venv\Lib\site-packages\beir\util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [ ]:
corpus, queries, qrels = data_loader("arguana")
cleaned_corpus, cleaned_queries = clean_data(corpus, queries)

100%|██████████| 8674/8674 [00:00<00:00, 161784.24it/s]


In [ ]:
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class LSAModel:
    def __init__(self, k_components="auto", use_query_expansion=True, expansion_nb=30,
                 alpha=1.0, beta=0.75, gamma=0.25, tau=0.9, min_k=100, max_k=700,
                 prefit_components=1000):

        self.vectorizer = TfidfVectorizer(sublinear_tf=True)
        self.k_components = k_components
        self.tau = tau
        self.min_k = min_k
        self.max_k = max_k
        self.prefit_components = prefit_components
        self.svd = None
        self.doc_vectors = None
        self.doc_ids = []
        self.doc_texts = []
        self.use_query_expansion = use_query_expansion
        self.expansion_nb = expansion_nb
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

    def select_k_by_variance(self, S):
        explained = np.cumsum(S**2) / np.sum(S**2)
        k = np.searchsorted(explained, self.tau) + 1
        k = min(max(k, self.min_k), self.max_k)
        return k

    def fit(self, corpus):
        self.doc_ids = list(corpus.keys())
        self.doc_texts = [doc["text"] for doc in corpus.values()]
        tfidf_matrix = self.vectorizer.fit_transform(self.doc_texts)

        if self.k_components == "auto":
            prefit = TruncatedSVD(
                k_components=min(self.prefit_components, tfidf_matrix.shape[1]-1),
                random_state=1
            )
            prefit.fit(tfidf_matrix)
            k = self.select_k_by_variance(prefit.singular_values_)
            print(f"[INFO] Nombre de composantes retenues : {k} (variance cumulative ≥ {self.tau*100:.0f}%)")
            self.svd = TruncatedSVD(n_components=k, random_state=1)
        else:
            self.svd = TruncatedSVD(n_components=self.n_components, random_state=1)

        self.doc_vectors = self.svd.fit_transform(tfidf_matrix)

    def search(self, query, top_k=10):
        query_tfidf = self.vectorizer.transform([query])
        query_lsa = self.svd.transform(query_tfidf)

        if self.use_query_expansion:
            sim_scores_init = cosine_similarity(query_lsa, self.doc_vectors)[0]

            top_indices_pos = np.argsort(sim_scores_init)[::-1][:self.expansion_nb]
            vecs_pos = self.doc_vectors[top_indices_pos]
            mean_pos = vecs_pos.mean(axis=0).reshape(1, -1)

            bottom_indices_neg = np.argsort(sim_scores_init)[:self.expansion_nb]
            vecs_neg = self.doc_vectors[bottom_indices_neg]
            mean_neg = vecs_neg.mean(axis=0).reshape(1, -1)

            query_lsa = (self.alpha * query_lsa +
                         self.beta * mean_pos -
                         self.gamma * mean_neg)

        scores = cosine_similarity(query_lsa, self.doc_vectors).flatten()
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.doc_ids[i], scores[i]) for i in top_indices]


In [ ]:
from beir.retrieval.evaluation import EvaluateRetrieval

def evaluate_model(model, queries, qrels, k_values=[10, 100]):
    results = {}
    max_k = max(k_values)

    for qid in qrels:
        if qid not in queries:
            continue
        query_text = queries[qid]["text"]
        scores = model.search(query_text, top_k=max_k)
        results[qid] = {doc_id: float(score) for doc_id, score in scores}

    ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(qrels, results, k_values=k_values)
    return (ndcg, _map, recall, precision)

